Copyright 2026 Snowflake Inc.  
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# From Messy Data to Instant Answers with Snowflake CoCo and CoWork

Northwind Outfitters sells shoes, clothing, accessories, and home goods online in the US, Canada, and Europe. Leadership wants to know:

> **Which products are customers returning most, and why?**

The data lives in three places that don't agree: an order export from the store, return events from three different systems (the returns portal, the warehouse, and customer support), and free-text comments customers left when they started a return. Some of those comments are in German, French, and Spanish.

In this lab you will:

1. Ask an AI agent the question over the raw data and get a confident, **wrong** answer.
2. Use **CoCo** to find out why, then fix what "a return" means and clean up the mess.
3. Use AI functions to translate customer comments and tag a reason on each one.
4. Give an agent both the numbers (Cortex Analyst) and the customers' own words (Cortex Search).
5. Ask the same question in **CoWork** and get an answer you can trust, with quotes.

The data is 2,000 order lines: big enough that you can't eyeball the answer, small enough that every step runs in seconds. The same steps work unchanged on millions of rows.

You'll see a lot of SQL. You don't need to understand every line: CoCo writes it, and each step ends with a check that tells you whether the numbers are right.

## Connect this notebook to compute

1. Click **Connect** next to the Run button and select **Create and connect**.
2. Set **Role = `ACCOUNTADMIN`** and **Warehouse = `COMPUTE_WH`**.
3. Run cells from top to bottom with **Run** or **Shift+Enter**.

Every object in this lab uses a fully qualified name, so you don't need to pick a database or schema.

In [ ]:
-- SETUP: create the lab database, schemas, and stage.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE ONLINE_RETURNS;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.RAW;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.CURATED;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.ANALYTICS;

CREATE OR REPLACE STAGE ONLINE_RETURNS.RAW.LAB_FILES;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Upload the raw files

The files in `data/` are messy on purpose. Upload them as they are, so the raw tables keep exactly what the source systems sent.

In [ ]:
from snowflake.snowpark.context import get_active_session
from pathlib import Path

session = get_active_session()

for filename in ["orders.csv", "return_events.json", "return_comments.json"]:
    matches = list(Path(".").glob(f"**/data/{filename}"))
    if not matches:
        raise FileNotFoundError(f"Could not find data/{filename}")
    session.file.put(
        str(matches[0]),
        "@ONLINE_RETURNS.RAW.LAB_FILES",
        auto_compress=False,
        overwrite=True,
    )
    print(f"Uploaded {filename}")

In [ ]:
-- Load the files into raw tables without changing anything.
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.ORDERS (
  order_line_id VARCHAR,
  order_id VARCHAR,
  customer_id VARCHAR,
  category VARCHAR,
  region VARCHAR,
  item_price VARCHAR,
  order_date VARCHAR,
  order_status VARCHAR
);
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.RETURN_EVENTS (payload VARIANT);
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.RETURN_COMMENTS (payload VARIANT);

COPY INTO ONLINE_RETURNS.RAW.ORDERS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('orders.csv')
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

COPY INTO ONLINE_RETURNS.RAW.RETURN_EVENTS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('return_events.json')
FILE_FORMAT = (TYPE = JSON);

COPY INTO ONLINE_RETURNS.RAW.RETURN_COMMENTS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('return_comments.json')
FILE_FORMAT = (TYPE = JSON);

-- Expect 2,000 order lines, 1,900 return events, 775 comments.
SELECT
  (SELECT COUNT(*) FROM ONLINE_RETURNS.RAW.ORDERS) AS order_lines,
  (SELECT ARRAY_SIZE(payload) FROM ONLINE_RETURNS.RAW.RETURN_EVENTS) AS return_events,
  (SELECT ARRAY_SIZE(payload) FROM ONLINE_RETURNS.RAW.RETURN_COMMENTS) AS comments;

## Part 1: The quick answer

The fastest thing you could do is point an AI agent straight at the order export and ask your question. The next cell sets that up with two Snowflake objects:

- A **semantic view** is a short description of your data in business terms ("an order line", "a returned line") that AI can use to write SQL for you.
- An **agent** is an AI assistant you can chat with in CoWork. This one answers questions by reading the semantic view.

This quick version takes the export at its word: a line counts as returned whenever `order_status` says `returned`. Run the cell. You don't need to understand every line of it.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW ONLINE_RETURNS.ANALYTICS.QUICK_RETURNS_SV
  TABLES (
    orders AS ONLINE_RETURNS.RAW.ORDERS PRIMARY KEY (order_line_id)
      COMMENT = 'Order lines exactly as exported from the store system'
  )
  DIMENSIONS (
    orders.category AS category,
    orders.region AS region,
    orders.order_status AS order_status
  )
  METRICS (
    orders.order_lines AS COUNT(order_line_id),
    orders.returned_lines AS COUNT_IF(LOWER(order_status) = 'returned'),
    return_rate AS DIV0(orders.returned_lines, orders.order_lines)
  )
  COMMENT = 'Quick semantic view over the raw export';

CREATE OR REPLACE AGENT ONLINE_RETURNS.ANALYTICS.QUICK_RETURNS_AGENT
  COMMENT = 'Quick agent over raw data. Deliberately not trustworthy.'
  FROM SPECIFICATION
$$
models:
  orchestration: auto
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: quick_returns
      description: Order lines from the raw store export.
tool_resources:
  quick_returns:
    semantic_view: ONLINE_RETURNS.ANALYTICS.QUICK_RETURNS_SV
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
$$;

### Ask the quick agent

Open **CoWork**, select `QUICK_RETURNS_AGENT`, and ask:

> *"Which product categories have the highest return rate?"*

Write down what it says. It will probably even clean up the category names for you (the export spells Footwear four different ways), and then tell you that **Apparel is the problem at about 45%** and **Footwear is the best performer at about 14%**.

It sounds confident, and it did some smart cleanup. It's still wrong, because it trusted what the export means by "returned". The rest of the lab shows why, and fixes it.

### Find the problem with CoCo

The export isn't the only record of returns. The returns portal, the warehouse, and customer support each log a return event as it moves along, in `RAW.RETURN_EVENTS`. Let CoCo compare the two. Open the **CoCo** panel and send:

> *"Compare the returns in ONLINE_RETURNS.RAW.ORDERS with the return events in ONLINE_RETURNS.RAW.RETURN_EVENTS. For each product category, how many lines does the export say were returned, and how many actually ended in a refund or exchange? Don't change any data."*

CoCo will probably also point out the messy category names and the different words the systems use for each status, without being asked. Read what it found, then compare with the expected output.

<details>
<summary>Expected output (check against this)</summary>

| Category | Export says returned | Events say refunded or exchanged |
|---|---:|---:|
| Accessories | 125 | 75 |
| Apparel | 250 | 150 |
| Footwear | **75** | **200** |
| Home | 125 | 75 |

</details>

Look at Footwear. The export says 75 returns; the events say 200. Every other category goes the *other* way. Something about Footwear returns never makes it back into the export.

### Look at two order lines up close

Run the next cell. It shows two order lines next to the full history of their return events.

In [ ]:
WITH events AS (
  SELECT
    value:order_line_id::VARCHAR AS order_line_id,
    value:event_timestamp::TIMESTAMP_TZ AS event_timestamp,
    value:return_status::VARCHAR AS return_status,
    value:source::VARCHAR AS source
  FROM ONLINE_RETURNS.RAW.RETURN_EVENTS,
       LATERAL FLATTEN(input => payload)
)
SELECT o.order_line_id, o.category, o.region, o.order_status AS export_says,
       e.event_timestamp, e.return_status, e.source
FROM ONLINE_RETURNS.RAW.ORDERS o
JOIN events e USING (order_line_id)
WHERE o.order_line_id IN ('L-2006', 'L-2020')
ORDER BY o.order_line_id, e.event_timestamp;

Things to notice:

- **L-2020 (Footwear, EMEA):** the export says `delivered`, but the events show the customer requested a return, the warehouse received it, and support issued a refund. It **is** a return. That's the Footwear gap you just found: Footwear returns are processed in a separate warehouse system that never updates the export, so the quick agent missed most of them.
- **L-2006 (Apparel):** the export says `Returned`, but the warehouse rejected the return. The customer kept it. It is **not** a return. The export marks a line "returned" as soon as someone *asks*.
- The three systems use different words for the same thing: `refund_issued` vs `refunded`, `exchange_shipped` vs `exchanged`, `Return Requested` vs `return_requested`.
- Some lines were never delivered (`shipped`, `cancelled`). They can't be returned, so they shouldn't count in the rate.

The fix is to agree on what "a return" means, and then build that rule into the data before any agent sees it.

### STEP 1: Define a return and clean the order lines

The rule for this lab:

- **A return** is an order line whose **latest** return event is a refund or an exchange. Requested or rejected returns don't count.
- **Return rate** = returned lines ÷ delivered lines.
- Categories and regions need one spelling each. Prices and dates need real types.

Open the **CoCo** panel and send this prompt. Compare CoCo's SQL with the expected output below. If it does the same thing, click **Allow** to run it. It doesn't have to match word for word.

> *"Create a clean table ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS with one row per order line from ONLINE_RETURNS.RAW.ORDERS. Use each line's latest return event. A line counts as returned only if that event is a refund or an exchange, and only delivered lines count toward the return rate (call the flags is_returned and is_delivered). Fix the messy category and region spellings, prices, and dates."*

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS AS
WITH events AS (
  SELECT
    value:order_line_id::VARCHAR        AS order_line_id,
    value:event_timestamp::TIMESTAMP_TZ AS event_timestamp,
    CASE LOWER(value:return_status::VARCHAR)
      WHEN 'return_requested' THEN 'requested'
      WHEN 'return requested' THEN 'requested'
      WHEN 'received'         THEN 'received'
      WHEN 'refunded'         THEN 'refunded'
      WHEN 'refund_issued'    THEN 'refunded'
      WHEN 'exchanged'        THEN 'exchanged'
      WHEN 'exchange_shipped' THEN 'exchanged'
      WHEN 'rejected'         THEN 'rejected'
      WHEN 'return_rejected'  THEN 'rejected'
    END AS return_status
  FROM ONLINE_RETURNS.RAW.RETURN_EVENTS,
       LATERAL FLATTEN(input => payload)
),
latest AS (
  SELECT order_line_id, return_status
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY event_timestamp DESC) = 1
)
SELECT
  o.order_line_id,
  o.order_id,
  o.customer_id,
  CASE
    WHEN LOWER(TRIM(o.category)) IN ('footwear', 'shoes')                        THEN 'Footwear'
    WHEN LOWER(TRIM(o.category)) IN ('apparel', 'clothing')                      THEN 'Apparel'
    WHEN LOWER(TRIM(o.category)) IN ('accessories', 'accessory', 'acc')          THEN 'Accessories'
    WHEN LOWER(TRIM(o.category)) IN ('home', 'home goods', 'home & living')      THEN 'Home'
  END AS category,
  CASE
    WHEN TRIM(o.region) IN ('US East', 'US-East', 'United States East', 'USA East') THEN 'US East'
    WHEN TRIM(o.region) IN ('US West', 'US-West', 'West Coast')                     THEN 'US West'
    WHEN TRIM(o.region) IN ('Canada', 'CA', 'CAN')                                  THEN 'Canada'
    WHEN TRIM(o.region) IN ('EMEA', 'Europe', 'Europe/ME/Africa')                   THEN 'EMEA'
  END AS region,
  TRY_TO_NUMBER(REPLACE(REPLACE(o.item_price, '$', ''), ',', ''), 12, 2) AS item_price,
  COALESCE(
    TRY_TO_DATE(o.order_date, 'MM/DD/YYYY'),
    TRY_TO_DATE(o.order_date, 'YYYY-MM-DD'),
    TRY_TO_DATE(o.order_date, 'MON DD YYYY'),
    TRY_TO_DATE(o.order_date, 'YYYY/MM/DD')
  ) AS order_dt,
  o.order_status AS source_order_status,
  LOWER(o.order_status) IN ('delivered', 'returned') AS is_delivered,
  l.return_status AS latest_return_status,
  COALESCE(l.return_status IN ('refunded', 'exchanged'), FALSE) AS is_returned
FROM ONLINE_RETURNS.RAW.ORDERS o
LEFT JOIN latest l USING (order_line_id);
```

</details>

### Check the numbers

Run this. You should see **2,000** order lines, **0** unmapped rows, **1,800** delivered, **500** returned, a **27.8%** return rate, and **$46,958.00** of returned merchandise. If any number is off, ask CoCo: *"Why doesn't this match 500 returned lines?"*

In [ ]:
SELECT
  COUNT(*)                                                        AS order_lines,
  COUNT_IF(category IS NULL OR region IS NULL OR item_price IS NULL OR order_dt IS NULL) AS unmapped_rows,
  COUNT_IF(is_delivered)                                          AS delivered_lines,
  COUNT_IF(is_returned)                                           AS returned_lines,
  ROUND(100 * DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 1) AS return_rate_pct,
  SUM(IFF(is_returned, item_price, 0))                            AS returned_value
FROM ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS;

### STEP 2: Translate the comments and tag a reason with AI

Customers explain their returns in their own words, and the European customers often write in German, French, or Spanish. Two AI functions turn that text into something you can count:

- `AI_TRANSLATE` turns each comment into English (an empty source language means "detect it for me").
- `AI_CLASSIFY` picks one reason from a fixed list.

Send this prompt to **CoCo**:

> *"Using the comments in ONLINE_RETURNS.RAW.RETURN_COMMENTS, create ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS that translates each comment to English with AI_TRANSLATE and tags its main return reason with AI_CLASSIFY (columns comment_english and return_reason): Wrong size, Damaged or defective, Not as described, Arrived late, Changed mind, or Other. Then create ONLINE_RETURNS.CURATED.RETURN_REVIEW joining the order lines to their comments."*

These calls run AI on 775 comments, so this cell takes a bit longer than a normal query (about 15–30 seconds).

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS AS
WITH comments AS (
  SELECT
    value:order_line_id::VARCHAR          AS order_line_id,
    value:comment_timestamp::TIMESTAMP_TZ AS comment_timestamp,
    value:comment::VARCHAR                AS comment
  FROM ONLINE_RETURNS.RAW.RETURN_COMMENTS,
       LATERAL FLATTEN(input => payload)
)
SELECT
  order_line_id,
  comment_timestamp,
  comment,
  AI_TRANSLATE(comment, '', 'en') AS comment_english,
  AI_CLASSIFY(
    comment,
    [
      {'label': 'Wrong size',           'description': 'The item is too big, too small, too tight, or does not fit'},
      {'label': 'Damaged or defective', 'description': 'The item arrived broken, scratched, or failed after little use'},
      {'label': 'Not as described',     'description': 'The color, material, or look does not match the listing or photos'},
      {'label': 'Arrived late',         'description': 'The item arrived too late to be useful'},
      {'label': 'Changed mind',         'description': 'Nothing is wrong with the item; the customer no longer wants it'},
      {'label': 'Other',                'description': 'No return reason given, such as a question about refund status'}
    ],
    {'task_description': 'Classify the main reason a customer gives for returning an online order. The comment may be in any language.'}
  ):labels[0]::STRING AS return_reason
FROM comments;

CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.RETURN_REVIEW AS
SELECT
  r.*,
  c.comment,
  c.comment_english,
  c.return_reason
FROM ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS r
LEFT JOIN ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS c USING (order_line_id);
```

</details>

### Look at what the AI decided

Run this and read a few rows. Check the translations of the non-English comments, and check that each reason makes sense. One comment, L-2043, mentions two different things (the photos and changing their mind). See if you agree with the label the AI picked.

In [ ]:
SELECT order_line_id, region, comment, comment_english, return_reason
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
WHERE comment IS NOT NULL
ORDER BY region, return_reason;

### Check the answer before handing it to an agent

If the numbers are wrong here, an agent will only make the wrong answer faster to get. Expect **2,000 / 1,800 / 500 / $46,958.00**, about **175** returned lines tagged `Wrong size`, and **0** invalid labels. AI labels can vary a little, so `Wrong size` may be off by one or two; a bigger gap means the labels or descriptions differ from the expected output.

In [ ]:
SELECT
  COUNT(*)                                             AS order_lines,
  COUNT_IF(is_delivered)                               AS delivered_lines,
  COUNT_IF(is_returned)                                AS returned_lines,
  SUM(IFF(is_returned, item_price, 0))                 AS returned_value,
  COUNT_IF(is_returned AND return_reason = 'Wrong size') AS wrong_size_returns,
  COUNT_IF(return_reason IS NOT NULL AND return_reason NOT IN
    ('Wrong size', 'Damaged or defective', 'Not as described', 'Arrived late', 'Changed mind', 'Other')) AS invalid_labels
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW;

### STEP 3: Give the agent your definitions (semantic view)

A **semantic view** is where you write down what your business words mean, such as "return rate" and "returned value", so every agent and every question uses the same definition. **Verified queries** are question-and-SQL pairs you've confirmed are correct. The agent uses them as worked examples.

Send this prompt to **CoCo**:

> *"Create semantic view ONLINE_RETURNS.ANALYTICS.RETURNS_SV over ONLINE_RETURNS.CURATED.RETURN_REVIEW. Return rate is returned lines divided by delivered lines, and returned value is the total price of returned lines. Let people slice by category, region, and return reason. Add verified queries for: return rate by category and region, overall return rate and value, and why customers return items."*

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE SEMANTIC VIEW ONLINE_RETURNS.ANALYTICS.RETURNS_SV
  TABLES (
    returns AS ONLINE_RETURNS.CURATED.RETURN_REVIEW
      PRIMARY KEY (order_line_id)
      COMMENT = 'One row per Q1 2026 order line. A return is a line whose latest return status is refunded or exchanged.'
  )
  DIMENSIONS (
    returns.category AS category COMMENT = 'Footwear, Apparel, Accessories, or Home',
    returns.region AS region COMMENT = 'US East, US West, Canada, or EMEA',
    returns.latest_return_status AS latest_return_status COMMENT = 'requested, received, refunded, exchanged, or rejected',
    returns.return_reason AS return_reason COMMENT = 'AI-derived reason from the customer comment. Other means no reason given.',
    returns.is_delivered AS is_delivered COMMENT = 'TRUE if the line was delivered (shipped and cancelled lines are FALSE)',
    returns.is_returned AS is_returned COMMENT = 'TRUE only if the latest return status is refunded or exchanged'
  )
  METRICS (
    returns.delivered_lines AS COUNT_IF(is_delivered) COMMENT = 'Delivered order lines, the denominator of the return rate',
    returns.returned_lines AS COUNT_IF(is_returned) COMMENT = 'Returned order lines (refunded or exchanged only)',
    returns.returned_value AS SUM(IFF(is_returned, item_price, 0)) COMMENT = 'USD value of returned order lines',
    return_rate AS DIV0(returns.returned_lines, returns.delivered_lines) COMMENT = 'Returned lines divided by delivered lines'
  )
  COMMENT = 'Northwind Outfitters returns, Q1 2026'
  AI_VERIFIED_QUERIES (
    rate_by_category_region AS (
      QUESTION 'What is our return rate by product category and region?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT category, region, COUNT_IF(is_delivered) AS delivered_lines, COUNT_IF(is_returned) AS returned_lines, ROUND(DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 3) AS return_rate FROM returns GROUP BY category, region ORDER BY return_rate DESC'
    ),
    overall_rate AS (
      QUESTION 'What is our overall return rate and how much revenue was returned?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT COUNT_IF(is_delivered) AS delivered_lines, COUNT_IF(is_returned) AS returned_lines, ROUND(DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 3) AS return_rate, SUM(IFF(is_returned, item_price, 0)) AS returned_value FROM returns'
    ),
    reasons AS (
      QUESTION 'Why are customers returning items?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT category, region, return_reason, COUNT(*) AS returned_lines, SUM(item_price) AS returned_value FROM returns WHERE is_returned GROUP BY category, region, return_reason ORDER BY returned_lines DESC, returned_value DESC'
    )
  );
```

</details>

In [ ]:
-- Expect 1,800 / 500 / 46,958.00 / 0.2778. Adjust metric names if yours differ.
SELECT * FROM SEMANTIC_VIEW(
  ONLINE_RETURNS.ANALYTICS.RETURNS_SV
  METRICS returns.delivered_lines, returns.returned_lines, returns.returned_value, return_rate
);

### STEP 4: Make the comments searchable (Cortex Search)

Numbers tell you *where* returns happen. Comments tell you *why*. A **Cortex Search** service lets an agent find the comments that match a question by meaning, not just by exact words, so "shoes too small" finds "Die Schuhe fallen eine ganze Nummer zu klein aus."

Send this prompt to **CoCo**:

> *"Create Cortex Search service ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH over the English comments in ONLINE_RETURNS.CURATED.RETURN_REVIEW, so I can filter by category, region, and return reason. Keep the original comment and order line ID too."*

The service takes a minute or so to build.

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE CORTEX SEARCH SERVICE ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH
  ON comment_english
  ATTRIBUTES category, region, return_reason, latest_return_status
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  AS (
    SELECT order_line_id, comment, comment_english, category, region, return_reason, latest_return_status
    FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
    WHERE comment IS NOT NULL
  );
```

</details>

In [ ]:
-- Expect wrong-size comments about shoes, several of them from EMEA in German, French, or Spanish.
SELECT r.value:order_line_id::STRING AS order_line_id,
       r.value:region::STRING AS region,
       r.value:comment::STRING AS comment,
       r.value:comment_english::STRING AS comment_english
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH',
  '{"query": "shoes too small", "columns": ["order_line_id", "comment", "comment_english", "region"], "limit": 5}'
)):results)) r;

### STEP 5: Build the agent with both tools

Now give one agent both tools: **Cortex Analyst** (over the semantic view) for the numbers, and **Cortex Search** for the customers' own words.

Send this prompt to **CoCo**:

> *"Create Cortex Agent ONLINE_RETURNS.ANALYTICS.RETURNS_AGENT that answers numbers questions with ONLINE_RETURNS.ANALYTICS.RETURNS_SV and finds customer quotes with ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH. When it quotes a customer, it should use their original words and add the English translation if needed. Start with the answer and keep it to the question asked."*

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE AGENT ONLINE_RETURNS.ANALYTICS.RETURNS_AGENT
  COMMENT = 'Northwind Outfitters returns agent'
  FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  orchestration: >
    Use returns_analytics for any count, rate, value, or ranking question.
    Use customer_comments to find what customers actually said, and quote them when you explain a reason.
  response: >
    A return is an order line whose latest return status is refunded or exchanged; requested and rejected
    returns are not returns. Return rate is returned lines divided by delivered lines. State counts,
    percentages, and USD values precisely. When you quote a customer, quote the COMMENT column exactly as
    written; if it is not in English, follow it with the English translation from COMMENT_ENGLISH in parentheses.
    Start the answer with the answer itself, not with notes about what you plan to do. Answer only the question asked.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: returns_analytics
      description: Return counts, rates, values, and AI-derived reasons by category and region for Q1 2026 order lines.
  - tool_spec:
      type: cortex_search
      name: customer_comments
      description: Customer return comments in the original language with an English translation, filterable by category, region, and reason.
tool_resources:
  returns_analytics:
    semantic_view: ONLINE_RETURNS.ANALYTICS.RETURNS_SV
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
  customer_comments:
    search_service: ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH
    max_results: 5
    id_column: ORDER_LINE_ID
    title_column: ORDER_LINE_ID
    columns_and_descriptions:
      COMMENT:
        description: "The customer's comment exactly as written, in its original language (English, German, French, or Spanish)."
        type: string
        searchable: false
        filterable: false
      COMMENT_ENGLISH:
        description: The comment translated into English. This is the searchable text.
        type: string
        searchable: true
        filterable: false
      REGION:
        description: "Customer region: US East, US West, Canada, or EMEA."
        type: string
        searchable: false
        filterable: true
      CATEGORY:
        description: "Product category: Footwear, Apparel, Accessories, or Home."
        type: string
        searchable: false
        filterable: true
$$;
```

</details>

## Part 2: Ask again in CoWork

Open **CoWork**, select `RETURNS_AGENT`, and ask the same question you asked the quick agent:

1. *"Which product categories have the highest return rate?"*
2. *"Which category and region should we look at first?"*
3. *"Why are customers returning footwear in EMEA? Show me what they said."*

**Expected:**

- Footwear has the highest return rate at **42.1%** (200 of 475 delivered lines), not the lowest. Apparel is second at **30.0%**.
- Overall: **500** of **1,800** delivered lines returned, **27.8%**, **$46,958.00**.
- Look first at **Footwear in EMEA**: **100 of 150** delivered lines returned (**66.7%**), and about **3 in 4** were **Wrong size**.
- The answer to question 3 should quote customers in their own language with a translation, such as *"Die Schuhe fallen eine ganze Nummer zu klein aus."* (*"The shoes are a whole size too small."*). It may mention a fifth sizing comment from a return that is only requested, so it doesn't count.

Now compare with what `QUICK_RETURNS_AGENT` told you at the start. Same question, same tool. The only difference is what you taught the data.

Try rewording question 1, for example *"What gets sent back most?"*, and check the answer doesn't change.

Optional: ask CoWork to turn the answer into a one-page brief for the merchandising team.

## What you built

- Raw tables that keep exactly what the source systems sent.
- A curated table with one clear definition of a return.
- AI-translated, AI-tagged customer comments you can count and search.
- A semantic view with verified queries, so every question uses the same definitions.
- An agent that answers with numbers *and* customer quotes.

The quick agent and the final agent used the same AI. What made the difference was the work you did on the data in between.

## Teardown

Run this only after you've finished the lab and any validation checks.

In [ ]:
DROP DATABASE IF EXISTS ONLINE_RETURNS;